In [1]:
import pandas as pd 
import numpy as np
import plotly.express as px

In [2]:
data = pd.read_excel('../data/get_around_delay_analysis.xlsx')
data.head(25)

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN
5,511626,398802,mobile,ended,-203.0,NaN,NaN
6,511639,370585,connect,ended,-15.0,563782.0,570.0
7,512303,371242,mobile,ended,-44.0,NaN,NaN
8,512475,322502,mobile,canceled,NaN,NaN,NaN
9,513434,256528,connect,ended,23.0,NaN,NaN


In [3]:
data_clean = data[(data["delay_at_checkout_in_minutes"].between(-1440, 1440))].copy()
data_clean = data_clean[data_clean['state'] == "ended"]
data_clean = data_clean.drop(columns=['car_id', 'rental_id', 'previous_ended_rental_id'])

data_clean.head()

,checkin_type,state,delay_at_checkout_in_minutes,time_delta_with_previous_rental_in_minutes
1,mobile,ended,-81.0,NaN
2,connect,ended,70.0,NaN
5,mobile,ended,-203.0,NaN
6,connect,ended,-15.0,570.0
7,mobile,ended,-44.0,NaN


# 1. Nombre de loc par rapport au dealy (retards, avance)

In [4]:
data_clean['delay_at_checkout_in_hours'] = data_clean['delay_at_checkout_in_minutes'] / 60

fig_hist_delay = px.histogram(
    data_clean,
    x="delay_at_checkout_in_hours",
    title="Distribution des retards/avances (±24h)",
    labels={"delay_at_checkout_in_hours": "Retard au check-out (minutes)"}
)

fig_hist_delay.add_vline(
    x=0,
    line_dash="dash",
    line_color="red",
    annotation_text="À l'heure",
    annotation_position="top left"
)

fig_hist_delay.update_layout(
    xaxis_title="Retard au check-out (heures)",
    yaxis_title="Nombre de locations",
    bargap=0.05
)
fig_hist_delay.update_traces(xbins=dict(start=-24, end=24, size=0.25))

In [5]:
data_clean.head()

,checkin_type,state,delay_at_checkout_in_minutes,time_delta_with_previous_rental_in_minutes,delay_at_checkout_in_hours
1,mobile,ended,-81.0,NaN,-1.350000
2,connect,ended,70.0,NaN,1.166667
5,mobile,ended,-203.0,NaN,-3.383333
6,connect,ended,-15.0,570.0,-0.250000
7,mobile,ended,-44.0,NaN,-0.733333


# 2. Pourcentage retard avance

In [6]:
def categorize_delay(x):
    if x > 5:
        return "retard"
    elif x < -5:
        return "avance"
    else:
        return "À l'heure"

data_clean["delay_category"] = data_clean["delay_at_checkout_in_minutes"].apply(categorize_delay)
delay_counts = data_clean["delay_category"].value_counts().reset_index()
delay_counts.columns = ["category", "count"]


In [7]:
fig = px.pie(
    delay_counts,
    names="category",
    values="count",
    color="category",
    title="Répartition des locations : retard vs avance",
    hole=0.3  # optionnel : donut chart
)

fig.show()

In [8]:
data_checkin_type_grouped = data_clean.groupby("checkin_type")["delay_at_checkout_in_hours"].value_counts().reset_index()
data_checkin_type_grouped

,checkin_type,delay_at_checkout_in_hours,count
0,connect,0.133333,34
1,connect,-0.100000,32
2,connect,0.116667,31
3,connect,-0.083333,30
4,connect,-0.033333,29
...,...,...,...
2120,mobile,23.666667,1
2121,mobile,23.700000,1
2122,mobile,23.783333,1
2123,mobile,23.900000,1


In [9]:
sum_by_checkin_type = data_clean.groupby("checkin_type")["delay_at_checkout_in_hours"].sum().reset_index()
type_connect_hours_delay_sum = sum_by_checkin_type["delay_at_checkout_in_hours"][0]
type_mobile_hours_delay_sum = sum_by_checkin_type["delay_at_checkout_in_hours"][1]
sum_by_checkin_type

,checkin_type,delay_at_checkout_in_hours
0,connect,-2289.083333
1,mobile,7941.150000


In [10]:
type_connect_hours_delay_sum = data_clean.loc[data_clean['checkin_type'] == "connect", "delay_at_checkout_in_hours"].sum()
type_mobile_hours_delay_sum = data_clean.loc[data_clean['checkin_type'] == "mobile", "delay_at_checkout_in_hours"].sum()
type_connect_hours_delay_sum

-2289.0833333333335

In [11]:
data_clean["delay_pct_by_type"] = (
    data_clean.groupby("checkin_type")["delay_at_checkout_in_hours"]
    .transform(lambda x: x / x.sum() * 100)
)

data_clean.sample(20)

,checkin_type,state,delay_at_checkout_in_minutes,time_delta_with_previous_rental_in_minutes,delay_at_checkout_in_hours,delay_category,delay_pct_by_type
7850,mobile,ended,10.0,NaN,0.166667,retard,0.002099
14740,connect,ended,68.0,NaN,1.133333,retard,-0.049510
17401,connect,ended,-1378.0,NaN,-22.966667,avance,1.003313
12822,mobile,ended,-1.0,NaN,-0.016667,À l'heure,-0.000210
14874,mobile,ended,7.0,NaN,0.116667,retard,0.001469
9712,connect,ended,-1.0,NaN,-0.016667,À l'heure,0.000728
17230,mobile,ended,-2.0,NaN,-0.033333,À l'heure,-0.000420
14549,mobile,ended,-25.0,NaN,-0.416667,avance,-0.005247
15032,connect,ended,-79.0,NaN,-1.316667,avance,0.057519
12046,mobile,ended,37.0,630.0,0.616667,retard,0.007765


In [12]:
df_decribe_by_type = data_clean.groupby("checkin_type")["delay_at_checkout_in_hours"].describe()
df_decribe_by_type

,count,mean,std,min,25%,50%,75%,max
checkin_type,,,,,,,,
connect,3393.0,-0.674649,3.232748,-23.966667,-1.25,-0.150000,0.533333,23.85
mobile,12724.0,0.624108,3.914084,-23.816667,-0.45,0.216667,1.216667,24.00


In [13]:
df_plot = df_decribe_by_type.T  # transpose
df_plot = df_plot.reset_index().rename(columns={"index": "stat"})

print(df_plot.head())

checkin_type   stat      connect        mobile
0             count  3393.000000  12724.000000
1              mean    -0.674649      0.624108
2               std     3.232748      3.914084
3               min   -23.966667    -23.816667
4               25%    -1.250000     -0.450000


In [14]:
df_melt = df_plot.melt(id_vars="stat", value_vars=["connect","mobile"],
            var_name="checkin_type", value_name="value")
df_melt = df_melt[df_melt['stat'] != 'count']
df_melt = df_melt[df_melt['stat'] != '25%']
df_melt = df_melt[df_melt['stat'] != '50%']
df_melt = df_melt[df_melt['stat'] != '75%']

print(df_melt.head())

   stat checkin_type      value
1  mean      connect  -0.674649
2   std      connect   3.232748
3   min      connect -23.966667
7   max      connect  23.850000
9  mean       mobile   0.624108


In [15]:
import plotly.express as px
#Mettr eun histogram par type

fig = px.bar(
    df_melt,
    x="stat",
    y="value",
    color="checkin_type",
    barmode="group",   # barres côte à côte
    title="Comparaison des stats descriptives par type"
)

fig.show()


# 3. Moyenne de temps par type de checkin

In [16]:
df_type_mean = data_clean.groupby("checkin_type")["delay_at_checkout_in_minutes"].mean().reset_index()
print(df_type_mean)

  checkin_type  delay_at_checkout_in_minutes
0      connect                    -40.478927
1       mobile                     37.446479


In [17]:
fig = px.bar(
    df_type_mean,
    x="checkin_type",
    y="delay_at_checkout_in_minutes"
)
fig.show()

# Pourcentage de conflit avec x h de buffer

In [18]:
conflict_percent_results = []

def add_scope(df_scope, scope_name):
    n_total = len(df_scope)
    for h in range(-24, 25):
        n_conflicts = (df_scope['delay_at_checkout_in_hours'] > h).sum()
        pct = round(100 * n_conflicts / n_total, 2) if n_total else 0.0
        conflict_percent_results.append({
            "buffer_hours": h,
            "conflict_percent": pct,
            "nbr_location_affect": int(n_conflicts),
            "type": scope_name
        })

add_scope(data_clean, "all")
add_scope(data_clean[data_clean['checkin_type'] == "mobile"], "mobile")
add_scope(data_clean[data_clean['checkin_type'] == "connect"], "connect")


df_buffer = pd.DataFrame(conflict_percent_results).sort_values(by="buffer_hours").reset_index()
df_buffer

,index,buffer_hours,conflict_percent,nbr_location_affect,type
0,0,-24,100.00,16117,all
1,49,-24,100.00,12724,mobile
2,98,-24,100.00,3393,connect
3,1,-23,99.93,16105,all
4,50,-23,99.92,12714,mobile
...,...,...,...,...,...
142,47,23,0.15,24,all
143,96,23,0.17,22,mobile
144,97,24,0.00,0,mobile
145,48,24,0.00,0,all


In [19]:
fig = px.line(
    df_buffer[df_buffer["type"] == "all"],
    x="buffer_hours",
    y="conflict_percent",
    markers=True
)
fig.show()

In [20]:
len(data_clean)

16117

In [21]:
len(data_clean[data_clean['delay_at_checkout_in_hours'] > 0])

9216

In [22]:
(len(data_clean[data_clean['delay_at_checkout_in_hours'] > 5])/len(data_clean))*100

5.342185270211578

In [23]:
data

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN
...,...,...,...,...,...,...,...
21305,573446,380069,mobile,ended,NaN,573429.0,300.0
21306,573790,341965,mobile,ended,-337.0,NaN,NaN
21307,573791,364890,mobile,ended,144.0,NaN,NaN
21308,574852,362531,connect,ended,-76.0,NaN,NaN


In [24]:
data[data["time_delta_with_previous_rental_in_minutes"].notna()]

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
6,511639,370585,connect,ended,-15.0,563782.0,570.0
19,519491,312389,mobile,ended,58.0,545639.0,420.0
23,521156,392479,mobile,ended,NaN,537298.0,0.0
34,525044,349751,mobile,ended,NaN,510607.0,60.0
40,528808,181625,connect,ended,-76.0,557404.0,330.0
...,...,...,...,...,...,...,...
21269,568049,381499,connect,canceled,NaN,562174.0,720.0
21272,568241,396409,mobile,canceled,NaN,566136.0,570.0
21275,568523,297973,mobile,ended,12.0,567121.0,240.0
21286,569717,377312,mobile,ended,230.0,545045.0,90.0


In [25]:
df_impact_late =  data[data["time_delta_with_previous_rental_in_minutes"].notna()].copy()
df_impact_late =  df_impact_late[df_impact_late["delay_at_checkout_in_minutes"].notna()]
df_impact_late["late_impact"] = round((df_impact_late["delay_at_checkout_in_minutes"] - df_impact_late["time_delta_with_previous_rental_in_minutes"])/60,2)
df_impact_late

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes,late_impact
6,511639,370585,connect,ended,-15.0,563782.0,570.0,-9.75
19,519491,312389,mobile,ended,58.0,545639.0,420.0,-6.03
40,528808,181625,connect,ended,-76.0,557404.0,330.0,-6.77
64,533670,320824,connect,ended,-6.0,556563.0,630.0,-10.60
74,534827,404169,mobile,ended,-7.0,531158.0,90.0,-1.62
...,...,...,...,...,...,...,...,...
21249,571823,353425,connect,ended,-276.0,569556.0,240.0,-8.60
21253,573274,298117,connect,ended,-7.0,571227.0,210.0,-3.62
21266,567741,294059,mobile,ended,111.0,567708.0,120.0,-0.15
21275,568523,297973,mobile,ended,12.0,567121.0,240.0,-3.80


In [26]:
round(len(df_impact_late)/len(data)*100, 2)

7.11

In [27]:
len(df_impact_late)

1515

In [28]:
mean_late_positive = df_impact_late.loc[df_impact_late["late_impact"] > 0, "late_impact"].mean()
print(round(mean_late_positive/60,2))


0.08


In [29]:
round((len(df_impact_late[df_impact_late['late_impact'] > 0])/len(df_impact_late))*100,2)

17.82

In [30]:
fig = px.histogram(
    df_impact_late[df_impact_late['late_impact'].between(0, 24)],
    x="late_impact",
    nbins=50,  # ajuste le nombre de barres
    labels={"late_impact": "Retard impactant (heure)", "count": "Nombre de cas"},
    title="Distribution des impacts de retard (>0)"
)
fig.show()

In [31]:
df_late_canceled =  data[data["time_delta_with_previous_rental_in_minutes"].notna()].copy()
len(df_late_canceled[df_late_canceled["state"] == "canceled"])

229

In [32]:
df_canceled = df_late_canceled[df_late_canceled["state"] == "canceled"]
df_canceled = df_canceled[df_canceled['time_delta_with_previous_rental_in_minutes']>0]
df_canceled

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
204,543768,374169,connect,canceled,NaN,543010.0,210.0
242,546160,352528,connect,canceled,NaN,546578.0,630.0
504,564627,341431,mobile,canceled,NaN,552005.0,150.0
637,568657,317378,connect,canceled,NaN,566412.0,210.0
669,516550,377700,mobile,canceled,NaN,545076.0,720.0
...,...,...,...,...,...,...,...
21022,560787,413181,mobile,canceled,NaN,560542.0,150.0
21172,566228,390871,connect,canceled,NaN,568465.0,60.0
21230,569706,245154,connect,canceled,NaN,558088.0,660.0
21269,568049,381499,connect,canceled,NaN,562174.0,720.0


In [33]:
round(len(df_canceled)/len(df_late_canceled[df_late_canceled["state"] == "canceled"])*100,2)

84.72

In [34]:
len(df_canceled)

194

In [35]:
data[
    (data["time_delta_with_previous_rental_in_minutes"].notna()) &
    (data["delay_at_checkout_in_minutes"] > data["time_delta_with_previous_rental_in_minutes"])
]

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
90,535770,352436,mobile,ended,74.0,524703.0,60.0
107,537576,397470,mobile,ended,18.0,539005.0,0.0
148,540479,374684,mobile,ended,12.0,539751.0,0.0
164,541862,382364,mobile,ended,125.0,540607.0,0.0
206,543808,369230,mobile,ended,75.0,536315.0,60.0
...,...,...,...,...,...,...,...
20922,561403,412139,mobile,ended,290.0,563942.0,120.0
20924,561476,410402,mobile,ended,11.0,550186.0,0.0
21054,562649,379751,connect,ended,72.0,565386.0,0.0
21163,565721,381470,mobile,ended,44.0,564855.0,0.0


In [36]:
len(data[(data["time_delta_with_previous_rental_in_minutes"].notna()) & (data["delay_at_checkout_in_minutes"] > data["time_delta_with_previous_rental_in_minutes"])])/len(data)*100

1.267010793054904

In [42]:
total_locations = len(data)

# Locations qui dépendent d'une précédente réservation
potentially_affected = data[data["previous_ended_rental_id"].notna()].copy()
potentially_affected_count = len(potentially_affected)

# % de revenu potentiel affecté
share_revenue_potentially_affected = potentially_affected_count / total_locations * 100

print(f"Nombre total de locations : {total_locations}")
print(f"Nombre de locations potentiellement affectées : {potentially_affected_count}")
print(f"% de revenu potentiellement affecté : {share_revenue_potentially_affected:.2f}%")


Nombre total de locations : 21310
Nombre de locations potentiellement affectées : 1841
% de revenu potentiellement affecté : 8.64%


In [43]:
potentially_affected["time_delta_with_previous_rental_in_hours"] = potentially_affected["time_delta_with_previous_rental_in_minutes"]/60
potentially_affected

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes,time_delta_with_previous_rental_in_hours
6,511639,370585,connect,ended,-15.0,563782.0,570.0,9.5
19,519491,312389,mobile,ended,58.0,545639.0,420.0,7.0
23,521156,392479,mobile,ended,NaN,537298.0,0.0,0.0
34,525044,349751,mobile,ended,NaN,510607.0,60.0,1.0
40,528808,181625,connect,ended,-76.0,557404.0,330.0,5.5
...,...,...,...,...,...,...,...,...
21269,568049,381499,connect,canceled,NaN,562174.0,720.0,12.0
21272,568241,396409,mobile,canceled,NaN,566136.0,570.0,9.5
21275,568523,297973,mobile,ended,12.0,567121.0,240.0,4.0
21286,569717,377312,mobile,ended,230.0,545045.0,90.0,1.5


In [45]:
fig = px.histogram(
    potentially_affected, 
    x="time_delta_with_previous_rental_in_hours",
    nbins=50,
    histnorm="percent"
)
fig.show()

In [46]:
counts, bins = np.histogram(
    potentially_affected["time_delta_with_previous_rental_in_hours"], bins=50
)

# Convertir en %
percentages = counts / len(data) * 100

# Mettre en DataFrame pour plotly
df_hist = pd.DataFrame({
    "bin": bins[:-1],
    "%_affected": percentages
})

fig = px.bar(df_hist, x="bin", y="%_affected")
fig.update_layout(
    xaxis_title="Buffer (heures)",
    yaxis_title="% du revenu potentiellement affecté (par rapport à toutes les locations)"
)
fig.show()

In [68]:
hour_impacted_revenue = []
print(share_revenue_potentially_affected)
for h in range(0,15):
    data_hour_revenue_count = len(potentially_affected[potentially_affected["time_delta_with_previous_rental_in_hours"]>=h])
    data_revenu_percentage_by_hour = round(data_hour_revenue_count / len(data)*100,2)
    hour_impacted_revenue.append(
        {
            "hours": h,
            "percentage": data_revenu_percentage_by_hour
        }
    )
    
print(hour_impacted_revenue)

8.639136555607696
[{'hours': 0, 'percentage': 8.64}, {'hours': 1, 'percentage': 6.76}, {'hours': 2, 'percentage': 5.51}, {'hours': 3, 'percentage': 4.56}, {'hours': 4, 'percentage': 3.94}, {'hours': 5, 'percentage': 3.45}, {'hours': 6, 'percentage': 3.15}, {'hours': 7, 'percentage': 2.89}, {'hours': 8, 'percentage': 2.59}, {'hours': 9, 'percentage': 2.22}, {'hours': 10, 'percentage': 1.84}, {'hours': 11, 'percentage': 1.25}, {'hours': 12, 'percentage': 0.61}, {'hours': 13, 'percentage': 0.0}, {'hours': 14, 'percentage': 0.0}]


In [71]:
df_impact_revenu_by_hours = pd.DataFrame(hour_impacted_revenue)
df_impact_revenu_by_hours["revert_percentage"] = round(share_revenue_potentially_affected - df_impact_revenu_by_hours["percentage"],2)
df_impact_revenu_by_hours

,hours,percentage,revert_percentage
0,0,8.64,-0.00
1,1,6.76,1.88
2,2,5.51,3.13
3,3,4.56,4.08
4,4,3.94,4.70
5,5,3.45,5.19
6,6,3.15,5.49
7,7,2.89,5.75
8,8,2.59,6.05
9,9,2.22,6.42


In [59]:
fig = px.line(
    df_impact_revenu_by_hours, 
    x="hour",
    y="percentage",
    markers=True
)
fig.show()

In [ ]:
hour_impacted_revenue = []
for h in range(0, 15):  # buffers en heures
    buffer_minutes = h * 60

    # Perte pour chaque location = min(buffer, delta réel)
    lost = potentially_affected["time_delta_with_previous_rental_in_minutes"].clip(upper=buffer_minutes)

    # Somme des pertes en minutes
    total_lost = lost.sum()

    # % par rapport au total de minutes observées
    pct_lost = total_lost / potentially_affected["time_delta_with_previous_rental_in_minutes"].sum() * 100

    hour_impacted_revenue.append({
        "hours": h,
        "percentage": round(pct_lost, 2)
    })

data_impact_revenu_by_hours = pd.DataFrame(hour_impacted_revenue)


In [65]:
data_impact_revenu_by_hours

,hours,percentage
0,0,0.00
1,1,422.62
2,2,764.99
3,3,1047.82
4,4,1291.08
5,5,1503.38
6,6,1695.96
7,7,1871.23
8,8,2029.61
9,9,2167.86
